# Baseline LSTM — Sequential Model

**Author:** Minho  
**Project:** Risk-Adjusted Portfolio Optimization  
**Model:** LSTM (sequential baseline)  
**Dataset:** `shared_set_2` (Growth/Tech/Innovation universe)  
**MLflow run name:** `Minho_Baseline_LSTM`

---

### What this notebook does

This is the simplest working LSTM baseline for the team's shared research framework.
It is intentionally kept minimal and readable — no tricks, no optimizations.

Steps:
1. Load shared dataset via `portfolio_toolkit`
2. Build shared toolkit features + 1 custom feature (`price_accel`)
3. Build per-ticker rolling sequences for the LSTM
4. Train a 2-layer LSTM to predict 5-day forward returns
5. Emit a standardized prediction table
6. Convert predictions → portfolio weights
7. Run the shared backtest
8. Log everything to MLflow as `Minho_Baseline_LSTM`

### How to run

```
cd <repo-root>
source venv312/bin/activate
jupyter notebook MODELS/Minho/baseline_lstm.ipynb
```

Run all cells top to bottom. No hidden state — every cell is self-contained.

## 0. Bootstrap — locate repo root

In [ ]:
import os
import sys
from pathlib import Path

# --- If auto-detection fails, set this manually ---
# repo_root = Path('/Users/minhochoi/Portfolio-Optimization-Lib')
# --------------------------------------------------

def _is_repo_root(p: Path) -> bool:
    return (p / 'pyproject.toml').exists() and (p / 'src' / 'portfolio_toolkit').exists()

if 'repo_root' not in dir() or not _is_repo_root(Path(repo_root)):
    _candidates = [Path.cwd(), *Path.cwd().parents]
    _found = next((p for p in _candidates if _is_repo_root(p)), None)
    if _found is None:
        raise RuntimeError(
            'Cannot find repo root. Uncomment and set repo_root manually at the top of this cell.'
        )
    repo_root = _found

repo_root = Path(repo_root).resolve()
os.chdir(repo_root)

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root:', repo_root)
print('python:   ', sys.executable)

## 1. Imports

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Config

All hyperparameters in one place.

In [ ]:
DATASET_NAME  = 'shared_set_2'
HORIZON       = 5          # forecast horizon in trading days
RUN_NAME      = 'Minho_Baseline_LSTM'

SEQ_LEN       = 20         # look-back window (trading days)
HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
DROPOUT       = 0.2
LR            = 1e-3
EPOCHS        = 30
BATCH_SIZE    = 512
SEED          = 42

artifact_dir  = repo_root / 'outputs' / RUN_NAME
artifact_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
print(f'Dataset : {DATASET_NAME}  ({len(spec.tickers)} tickers)')
print(f'Train   : {spec.train_start} → {spec.train_end}')
print(f'Val     : {spec.val_start}   → {spec.val_end}')
print(f'Test    : {spec.test_start}  → {spec.test_end}')

## 3. Load Prices

In [ ]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print('Shape     :', prices.shape)
print('Date range:', prices['date'].min().date(), '→', prices['date'].max().date())
prices.head(3)

## 4. Features — Shared Toolkit + 1 Custom

**Custom feature: `price_accel`**  
Defined as `return_5d − return_20d`. Captures whether short-term momentum is
accelerating (positive) or decelerating (negative) relative to the medium-term trend.
Particularly informative for an LSTM because the model can learn how acceleration
evolves across the look-back window.

In [ ]:
BASE_FEATURES = [
    'return_1d', 'return_5d', 'return_20d',
    'vol_5d', 'vol_20d',
    'momentum_5d', 'momentum_20d', 'momentum_60d',
    'rsi_14', 'macd_hist', 'bollinger_z_20d',
    'beta_20d_spy', 'excess_return_5d_vs_spy',
    'volume_zscore_20d', 'price_to_sma_20d', 'atr_14',
]

feature_frame = build_features(prices, feature_names=BASE_FEATURES)

# --- Custom feature: price acceleration ---
panel = prices.sort_values(['ticker', 'date'])
ret5  = panel.groupby('ticker')['adj_close'].pct_change(5)
ret20 = panel.groupby('ticker')['adj_close'].pct_change(20)

custom_df = panel[['date', 'ticker']].copy()
custom_df['price_accel'] = (ret5 - ret20).values

frame = feature_frame.merge(custom_df, on=['date', 'ticker'], how='left')
ALL_FEATURES = BASE_FEATURES + ['price_accel']

print(f'Features: {len(ALL_FEATURES)}  ({len(BASE_FEATURES)} shared + 1 custom)')
frame[['date', 'ticker', 'price_accel']].dropna().head(3)

## 5. Targets + Train / Val / Test Split

In [ ]:
TARGET_COL = f'forward_return_{HORIZON}d'

targets      = make_forward_return_target(prices, horizon=HORIZON)
target_frame = frame.merge(targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

train = slice_split(target_frame, DATASET_NAME, 'train', repo_root=repo_root)
val   = slice_split(target_frame, DATASET_NAME, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, DATASET_NAME, 'test',  repo_root=repo_root)

# Normalize using train statistics only (no leakage)
train_mean = train[ALL_FEATURES].mean()
train_std  = train[ALL_FEATURES].std(ddof=0).replace(0.0, 1.0)

def scale(df):
    return ((df[ALL_FEATURES] - train_mean) / train_std).to_numpy(dtype=np.float32)

X_train, X_val, X_test = scale(train), scale(val), scale(test)
y_train = train[TARGET_COL].to_numpy(dtype=np.float32)
y_val   = val[TARGET_COL].to_numpy(dtype=np.float32)

print(f'Train : {len(train):>6,} rows    X_train : {X_train.shape}')
print(f'Val   : {len(val):>6,} rows    X_val   : {X_val.shape}')
print(f'Test  : {len(test):>6,} rows    X_test  : {X_test.shape}')

## 6. Build Per-Ticker Sequences

LSTMs consume `(batch, seq_len, n_features)` tensors. We build overlapping
windows of length `SEQ_LEN` **within each ticker** so sequences never span
ticker boundaries. The label is the target at the last timestep of each window.

In [ ]:
def make_sequences(
    df: pd.DataFrame,
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
):
    """Return (X_seq, y_seq, meta) where meta is list of (date, ticker)."""
    Xs, ys, meta = [], [], []
    df = df.reset_index(drop=True)
    for ticker, grp in df.groupby('ticker', sort=False):
        idx   = grp.index.tolist()
        dates = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            Xs.append(X[rows])
            ys.append(y[idx[end]])
            meta.append((dates[end], ticker))
    return (
        np.stack(Xs).astype(np.float32),
        np.array(ys, dtype=np.float32),
        meta,
    )

print('Building sequences ...')
X_tr_seq, y_tr_seq, _          = make_sequences(train, X_train, y_train, SEQ_LEN)
X_va_seq, y_va_seq, _          = make_sequences(val,   X_val,   y_val,   SEQ_LEN)
X_te_seq, y_te_seq, meta_test  = make_sequences(test,  X_test,  test[TARGET_COL].to_numpy(np.float32), SEQ_LEN)

print(f'Train sequences : {X_tr_seq.shape}')   # (N, SEQ_LEN, F)
print(f'Val sequences   : {X_va_seq.shape}')
print(f'Test sequences  : {X_te_seq.shape}')

## 7. LSTM Model

A minimal 2-layer stacked LSTM. Input → LSTM → last hidden state → linear head → scalar return prediction.
Causal (not bidirectional). Gradient clipping applied during training.

In [ ]:
class BaselineLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)       # h_n: (num_layers, batch, hidden)
        return self.head(h_n[-1]).squeeze(-1)  # scalar per sequence


def train_lstm(model, X_tr, y_tr, X_va, y_va, epochs, batch_size, lr, device):
    """Train and return loss history DataFrame."""
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    loader    = DataLoader(
        TensorDataset(
            torch.as_tensor(X_tr, dtype=torch.float32),
            torch.as_tensor(y_tr, dtype=torch.float32),
        ),
        batch_size=batch_size, shuffle=True,
    )
    history = []
    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            Xt = torch.as_tensor(X_tr, dtype=torch.float32, device=device)
            tr_loss = float(loss_fn(model(Xt), torch.as_tensor(y_tr, device=device)))
            Xv = torch.as_tensor(X_va, dtype=torch.float32, device=device)
            va_loss = float(loss_fn(model(Xv), torch.as_tensor(y_va, device=device)))

        history.append({'epoch': epoch + 1, 'train_loss': tr_loss, 'val_loss': va_loss})
        if (epoch + 1) % 5 == 0:
            print(f'  epoch {epoch+1:3d}/{epochs}  train={tr_loss:.6f}  val={va_loss:.6f}')

    return pd.DataFrame(history)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(f'Input size: {X_tr_seq.shape[2]}')

## 8. Train

In [ ]:
model = BaselineLSTM(
    input_size  = X_tr_seq.shape[2],
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
)

print(f'Training on {X_tr_seq.shape[0]:,} sequences for {EPOCHS} epochs ...')
history = train_lstm(
    model, X_tr_seq, y_tr_seq, X_va_seq, y_va_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
)

best_val = history['val_loss'].min()
print(f'\nBest val MSE: {best_val:.8f}')
history.tail()

## 9. Predictions

In [ ]:
model.eval()
with torch.no_grad():
    raw = model(
        torch.as_tensor(X_te_seq, dtype=torch.float32, device=DEVICE)
    ).cpu().numpy()

pred_dates, pred_tickers = zip(*meta_test)
predictions = pd.DataFrame({
    'date'            : pd.to_datetime(list(pred_dates)),
    'ticker'          : list(pred_tickers),
    'horizon'         : HORIZON,
    'expected_return' : raw.astype(float),
})
predictions = predictions.drop_duplicates(['date', 'ticker', 'horizon'], keep='last')
predictions = validate_prediction_frame(
    predictions, dataset_name=DATASET_NAME, repo_root=repo_root
)

print('Predictions shape:', predictions.shape)
print('Date range:', predictions['date'].min().date(), '→', predictions['date'].max().date())
predictions.head()

## 10. Portfolio Weights

In [ ]:
portfolio = weights_from_predictions_rank_long_only(
    predictions,
    score_column  = 'expected_return',
    dataset_name  = DATASET_NAME,
    strategy_name = RUN_NAME,
)

validated_weights = validate_weights_frame(
    portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root
)

print('Weights shape:', validated_weights.shape)
print('Row sums (all should be 1.0):')
print(validated_weights.sum(axis=1).describe())

## 11. Backtest

In [ ]:
result = backtest_weights(
    DATASET_NAME, portfolio, benchmark='SPY', repo_root=repo_root
)

print('Backtest metrics:')
for k, v in sorted(result.metrics.items()):
    print(f'  {k:<35s}: {v:.6f}')

## 12. Write Artifacts (QuantStats report + parquet files)

In [ ]:
artifact_paths = write_backtest_artifacts(result, artifact_dir)

for key, path in artifact_paths.items():
    exists = '✓' if Path(path).exists() else '✗'
    print(f'  {exists} {key:<20s}: {path}')

## 13. Log to MLflow as `Minho_Baseline_LSTM`

In [ ]:
import mlflow

mlflow_layout = init_mlflow(repo_root)
print('Tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name     = RUN_NAME,
    dataset_name = DATASET_NAME,
    tags={
        'author'       : 'Minho',
        'model_family' : 'sequential_lstm',
        'workflow'     : 'baseline_lstm',
        'horizon'      : str(HORIZON),
        'project'      : 'risk_adjusted_portfolio_optimization',
    },
    repo_root=repo_root,
):
    mlflow.log_params({
        'run_name'      : RUN_NAME,
        'dataset'       : DATASET_NAME,
        'horizon'       : HORIZON,
        'seq_len'       : SEQ_LEN,
        'hidden_size'   : HIDDEN_SIZE,
        'num_layers'    : NUM_LAYERS,
        'dropout'       : DROPOUT,
        'lr'            : LR,
        'epochs'        : EPOCHS,
        'batch_size'    : BATCH_SIZE,
        'n_features'    : len(ALL_FEATURES),
        'custom_features': 'price_accel',
        'portfolio_builder': 'rank_long_only',
        'best_val_mse'  : round(float(best_val), 8),
    })
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print(f'MLflow run "{RUN_NAME}" logged successfully.')

## 14. Final Checks

In [ ]:
from IPython.display import display

# Sanity assertions
assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)
assert 'price_accel' in frame.columns

print('All checks passed. Notebook ran end to end successfully.')
print()
print('Key metrics:')
for k in ['annual_return', 'sharpe', 'max_drawdown', 'average_turnover']:
    print(f'  {k:<20s}: {result.metrics[k]:.4f}')

print()
display(result.nav.tail(3).to_frame('nav'))

## Secret Set 2 Backtest: Minho Baseline LSTM v2

Standalone cells for rerunning Minho's newest `sp500_subset_50` MLflow upload on `secret_set_2`. These cells do not depend on any earlier notebook state. They use the v2 MLflow-downloaded notebook and checkpoint, rebuild Minho's LSTM feature pipeline exactly from this submitted source (`build_features` plus `price_accel`), use SPY only as feature-engineering context, filter predictions back to the `secret_set_2` tradable universe, rebalance every 5 trading days as recorded in the checkpoint, and backtest against the dataset benchmark (`XBI`).


In [1]:
# Secret Set 2 standalone setup for Minho's newest sp500_subset_50 LSTM v2 upload
import json
import os
import random
import re
import shutil
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

warnings.filterwarnings("ignore")


def _is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if _is_repo_root(candidate):
            return candidate.resolve()
    raise RuntimeError("Could not find repo root. Run this notebook from inside Portfolio-Optimizer.")


repo_root = _find_repo_root()
os.chdir(repo_root)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    build_metrics,
    custom_dataset,
    get_dataset_spec,
    init_mlflow,
    load_mlflow_settings,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    start_run,
    validate_feature_frame,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

MINHO_V2_SOURCE_RUN_ID = "5724af17ff454466b27fe22f89fdd2a4"
MINHO_V2_SOURCE_EXPERIMENT = "portfolio_toolkit_sp500_subset_50"
MINHO_V2_ARTIFACT_DIR = repo_root / "runs" / "minho_sp500_subset_50_v2_artifacts"
MINHO_V2_MODEL_DIR = repo_root / "MODELS" / "Minho"
MINHO_V2_NOTEBOOK_PATH = MINHO_V2_MODEL_DIR / "baseline_lstm_v2.ipynb"
MINHO_V2_CHECKPOINT_PATH = MINHO_V2_MODEL_DIR / "lstm_checkpoint_v2.pt"
MINHO_V2_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MINHO_V2_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MINHO_V2_REMOTE_ARTIFACTS = {
    "checkpoint": "model_submission/artifacts/lstm_checkpoint.pt",
    "manifest": "model_submission/manifest.json",
    "source_notebook": "model_submission/source/baseline_lstm.ipynb",
    "source_weights": "weights.parquet",
    "source_predictions": "predictions.parquet",
    "source_dataset_spec": "dataset_spec.json",
}
MINHO_V2_LOCAL_ARTIFACTS = {
    "checkpoint": MINHO_V2_ARTIFACT_DIR / "lstm_checkpoint_v2.pt",
    "manifest": MINHO_V2_ARTIFACT_DIR / "manifest_v2.json",
    "source_notebook": MINHO_V2_ARTIFACT_DIR / "baseline_lstm_v2.ipynb",
    "source_weights": MINHO_V2_ARTIFACT_DIR / "source_weights_v2.parquet",
    "source_predictions": MINHO_V2_ARTIFACT_DIR / "source_predictions_v2.parquet",
    "source_dataset_spec": MINHO_V2_ARTIFACT_DIR / "source_dataset_spec_v2.json",
}


def _ensure_minho_v2_mlflow_artifacts() -> dict[str, Path]:
    missing = [name for name, path in MINHO_V2_LOCAL_ARTIFACTS.items() if not path.exists()]
    if missing:
        import mlflow
        from mlflow.tracking import MlflowClient

        mlflow_layout = init_mlflow(repo_root=repo_root)
        mlflow.set_tracking_uri(mlflow_layout["tracking_uri"])
        client = MlflowClient()
        download_dir = MINHO_V2_ARTIFACT_DIR / "_downloaded"
        for name in missing:
            downloaded = Path(client.download_artifacts(MINHO_V2_SOURCE_RUN_ID, MINHO_V2_REMOTE_ARTIFACTS[name], str(download_dir)))
            local_path = MINHO_V2_LOCAL_ARTIFACTS[name]
            local_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(downloaded, local_path)

    if not MINHO_V2_CHECKPOINT_PATH.exists():
        shutil.copy2(MINHO_V2_LOCAL_ARTIFACTS["checkpoint"], MINHO_V2_CHECKPOINT_PATH)
    if not MINHO_V2_NOTEBOOK_PATH.exists():
        shutil.copy2(MINHO_V2_LOCAL_ARTIFACTS["source_notebook"], MINHO_V2_NOTEBOOK_PATH)
    return dict(MINHO_V2_LOCAL_ARTIFACTS)


minho_v2_artifact_paths = _ensure_minho_v2_mlflow_artifacts()
MINHO_V2_SOURCE_MANIFEST_PATH = minho_v2_artifact_paths["manifest"]
MINHO_V2_SOURCE_NOTEBOOK_ARTIFACT_PATH = minho_v2_artifact_paths["source_notebook"]
MINHO_V2_SOURCE_WEIGHTS_PATH = minho_v2_artifact_paths["source_weights"]
MINHO_V2_SOURCE_PREDICTIONS_PATH = minho_v2_artifact_paths["source_predictions"]
MINHO_V2_SOURCE_DATASET_SPEC_PATH = minho_v2_artifact_paths["source_dataset_spec"]

checkpoint = torch.load(MINHO_V2_CHECKPOINT_PATH, map_location="cpu")
if "state_dict" not in checkpoint:
    raise KeyError(f"Expected state_dict in checkpoint: {MINHO_V2_CHECKPOINT_PATH}")

SECRET_DATASET_NAME = "secret_set_2"
MINHO_V2_MODEL_NAME = str(checkpoint.get("run_name", "Minho_Baseline_LSTM_v2"))
MINHO_V2_FEATURE_NAMES = list(checkpoint["feature_names"])
MINHO_V2_BASE_FEATURES = [name for name in MINHO_V2_FEATURE_NAMES if name != "price_accel"]
MINHO_V2_HORIZON = int(checkpoint.get("horizon", 5))
MINHO_V2_MODEL_CONFIG = dict(checkpoint.get("model_config", {}))
MINHO_V2_PREPROCESSING = dict(checkpoint.get("preprocessing", {}))
MINHO_V2_DATA_METADATA = dict(checkpoint.get("data", {}))
MINHO_V2_TRAINING_METADATA = dict(checkpoint.get("training", {}))
MINHO_V2_REBALANCE_FREQUENCY = str(checkpoint.get("rebalance_frequency", "every_5_trading_days"))
MINHO_V2_SEQ_LEN = int(MINHO_V2_DATA_METADATA.get("seq_len", 20))
MINHO_V2_SOURCE_DATASET_NAME = str(MINHO_V2_DATA_METADATA.get("dataset_name", "sp500_subset_50"))
FEATURE_BENCHMARK_TICKER = str(MINHO_V2_DATA_METADATA.get("benchmark", "SPY")).upper()

secret_spec = get_dataset_spec(SECRET_DATASET_NAME, repo_root=repo_root)
backtest_benchmark_ticker = secret_spec.default_benchmark.upper()
DEVICE = torch.device("cpu")
MINHO_V2_OUTPUT_DIR = repo_root / "runs" / "minho_baseline_lstm_v2_secret_set_2_backtest"
MINHO_V2_MODEL_BUNDLE_DIR = MINHO_V2_OUTPUT_DIR / "model_bundle"

random.seed(int(MINHO_V2_TRAINING_METADATA.get("seed", 42)))
np.random.seed(int(MINHO_V2_TRAINING_METADATA.get("seed", 42)))
torch.manual_seed(int(MINHO_V2_TRAINING_METADATA.get("seed", 42)))

print("repo_root:", repo_root)
print("source MLflow experiment:", MINHO_V2_SOURCE_EXPERIMENT)
print("source MLflow run:", MINHO_V2_SOURCE_RUN_ID)
print("v2 notebook:", MINHO_V2_NOTEBOOK_PATH)
print("v2 checkpoint:", MINHO_V2_CHECKPOINT_PATH)
print("source dataset:", MINHO_V2_SOURCE_DATASET_NAME)
print("secret dataset:", SECRET_DATASET_NAME, secret_spec.name)
print("feature benchmark:", FEATURE_BENCHMARK_TICKER)
print("backtest benchmark:", backtest_benchmark_ticker)
print("rebalance frequency:", MINHO_V2_REBALANCE_FREQUENCY)
print("features:", len(MINHO_V2_FEATURE_NAMES))
print("device:", DEVICE)


repo_root: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
source MLflow experiment: portfolio_toolkit_sp500_subset_50
source MLflow run: 5724af17ff454466b27fe22f89fdd2a4
v2 notebook: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/MODELS/Minho/baseline_lstm_v2.ipynb
v2 checkpoint: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/MODELS/Minho/lstm_checkpoint_v2.pt
source dataset: sp500_subset_50
secret dataset: secret_set_2 XBI
feature benchmark: SPY
backtest benchmark: XBI
rebalance frequency: every_5_trading_days
features: 17
device: cpu


In [2]:
# Minho v2 LSTM feature, sequence, and inference helpers from the submitted MLflow notebook
def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    feature_frame = build_features(prices, feature_names=MINHO_V2_BASE_FEATURES)

    panel = prices.sort_values(["ticker", "date"]).copy()
    ret5 = panel.groupby("ticker")["adj_close"].pct_change(5)
    ret20 = panel.groupby("ticker")["adj_close"].pct_change(20)
    custom_df = panel[["date", "ticker"]].copy()
    custom_df["price_accel"] = (ret5 - ret20).to_numpy(dtype=float)

    frame = feature_frame.merge(custom_df, on=["date", "ticker"], how="left")
    return validate_feature_frame(frame.loc[:, ["date", "ticker", *MINHO_V2_FEATURE_NAMES]])


class BaselineLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1]).squeeze(-1)


def _checkpoint_scaler() -> tuple[pd.Series, pd.Series]:
    means = MINHO_V2_PREPROCESSING.get("means", {})
    stds = MINHO_V2_PREPROCESSING.get("stds", {})
    train_mean = pd.Series(means, dtype=float).reindex(MINHO_V2_FEATURE_NAMES)
    train_std = pd.Series(stds, dtype=float).reindex(MINHO_V2_FEATURE_NAMES).replace(0.0, 1.0)
    if train_mean.isna().any() or train_std.isna().any():
        missing = sorted(
            name
            for name in MINHO_V2_FEATURE_NAMES
            if name not in means or name not in stds or pd.isna(train_mean.loc[name]) or pd.isna(train_std.loc[name])
        )
        raise ValueError(f"Checkpoint scaler is missing feature stats: {missing}")
    return train_mean, train_std


def _select_rebalance_dates(dates, frequency: str) -> pd.DatetimeIndex:
    available = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates).drop_duplicates())).tz_localize(None).sort_values()
    if available.empty:
        raise ValueError("No prediction dates are available for rebalance selection")

    cleaned = str(frequency).strip().lower()
    if cleaned in {"daily", "every_1_trading_day", "every_1_trading_days"}:
        return available
    if cleaned == "none":
        return pd.DatetimeIndex([available[0]])
    if cleaned == "weekly":
        selected = available.to_series(index=available).groupby(available.to_period("W-FRI")).first()
        return pd.DatetimeIndex(selected.to_numpy()).tz_localize(None)
    if cleaned == "monthly":
        selected = available.to_series(index=available).groupby(available.to_period("M")).first()
        return pd.DatetimeIndex(selected.to_numpy()).tz_localize(None)

    match = re.fullmatch(r"every_(\d+)_trading_days?", cleaned)
    if match:
        step = int(match.group(1))
        if step <= 0:
            raise ValueError(f"Invalid rebalance frequency: {frequency}")
        return available[::step]

    raise ValueError(f"Unsupported rebalance frequency: {frequency}")


def _predict_from_feature_frame(
    model: nn.Module,
    feature_frame: pd.DataFrame,
    train_mean: pd.Series,
    train_std: pd.Series,
    *,
    batch_size: int = 4096,
) -> pd.DataFrame:
    frame = (
        feature_frame.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=MINHO_V2_FEATURE_NAMES)
        .sort_values(["ticker", "date"])
        .reset_index(drop=True)
    )
    if frame.empty:
        raise ValueError("No rows remain after feature construction and NaN filtering")

    mean = train_mean.reindex(MINHO_V2_FEATURE_NAMES).astype(float)
    std = train_std.reindex(MINHO_V2_FEATURE_NAMES).astype(float).replace(0.0, 1.0)
    X_scaled = ((frame[MINHO_V2_FEATURE_NAMES] - mean) / std).to_numpy(dtype=np.float32)

    X_sequences, meta = [], []
    for ticker, group in frame.groupby("ticker", sort=False):
        idx = group.index.to_numpy()
        dates = pd.to_datetime(group["date"]).to_numpy()
        if len(idx) < MINHO_V2_SEQ_LEN:
            continue
        for end in range(MINHO_V2_SEQ_LEN - 1, len(idx)):
            rows = idx[end - MINHO_V2_SEQ_LEN + 1 : end + 1]
            X_sequences.append(X_scaled[rows])
            meta.append((pd.Timestamp(dates[end]), ticker))

    if not X_sequences:
        raise ValueError("No LSTM sequences could be built from the feature frame")

    X_array = np.stack(X_sequences).astype(np.float32)
    raw_predictions = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X_array), batch_size):
            X_batch = torch.as_tensor(
                X_array[start : start + batch_size],
                dtype=torch.float32,
                device=next(model.parameters()).device,
            )
            raw_predictions.append(model(X_batch).detach().cpu().numpy())

    pred_values = np.concatenate(raw_predictions).astype(float)
    pred_dates, pred_tickers = zip(*meta)
    predictions = pd.DataFrame(
        {
            "date": pd.to_datetime(list(pred_dates)),
            "ticker": list(pred_tickers),
            "horizon": MINHO_V2_HORIZON,
            "expected_return": pred_values,
        }
    ).drop_duplicates(["date", "ticker", "horizon"], keep="last")
    return validate_prediction_frame(predictions, horizon=MINHO_V2_HORIZON)


print("Minho v2 Secret Set 2 helper functions ready.")


Minho v2 Secret Set 2 helper functions ready.


In [3]:
# Load Minho v2's checkpoint and rebuild Secret Set 2 features with SPY context only
train_mean, train_std = _checkpoint_scaler()

model = BaselineLSTM(
    input_size=len(MINHO_V2_FEATURE_NAMES),
    hidden_size=int(MINHO_V2_MODEL_CONFIG.get("hidden_dim", MINHO_V2_MODEL_CONFIG.get("hidden_size", 64))),
    num_layers=int(MINHO_V2_MODEL_CONFIG.get("num_layers", 2)),
    dropout=float(MINHO_V2_MODEL_CONFIG.get("dropout", 0.2)),
)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

secret_prices = load_prices(SECRET_DATASET_NAME, repo_root=repo_root)
secret_universe_prices = secret_prices.loc[secret_prices["ticker"].isin(secret_spec.tickers)].copy()

spy_context_dataset = custom_dataset(
    tickers=[FEATURE_BENCHMARK_TICKER],
    start=secret_spec.start_date,
    end=secret_spec.end_date,
    benchmark=FEATURE_BENCHMARK_TICKER,
    name="minho_v2_secret_set_2_spy_feature_context",
    cost_bps=secret_spec.cost_bps,
)
spy_prices = load_prices(spy_context_dataset, repo_root=repo_root)
spy_prices = spy_prices.loc[spy_prices["ticker"] == FEATURE_BENCHMARK_TICKER].copy()
if spy_prices.empty:
    raise ValueError(f"{FEATURE_BENCHMARK_TICKER} rows are missing; cannot build Minho's SPY-relative features")

feature_prices = (
    pd.concat([secret_universe_prices, spy_prices], ignore_index=True)
    .drop_duplicates(["date", "ticker"], keep="last")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)
all_features = build_model_features(feature_prices)
secret_features = all_features.loc[all_features["ticker"].isin(secret_spec.tickers)].copy()

full_predictions = _predict_from_feature_frame(model, secret_features, train_mean, train_std)
test_predictions = full_predictions.loc[
    (full_predictions["date"] >= pd.Timestamp(secret_spec.test_start))
    & (full_predictions["date"] <= pd.Timestamp(secret_spec.test_end))
].copy()
if test_predictions.empty:
    raise ValueError("No predictions were generated inside the Secret Set 2 test window")

rebalance_dates = _select_rebalance_dates(test_predictions["date"], MINHO_V2_REBALANCE_FREQUENCY)
predictions = test_predictions.loc[test_predictions["date"].isin(rebalance_dates)].copy()
predictions = validate_prediction_frame(
    predictions,
    dataset_name=SECRET_DATASET_NAME,
    horizon=MINHO_V2_HORIZON,
    repo_root=repo_root,
)

print("secret price rows:", len(secret_universe_prices))
print("SPY context rows:", len(spy_prices))
print("feature rows:", len(secret_features))
print("all sequence predictions:", full_predictions.shape)
print("test prediction dates before rebalance filter:", test_predictions["date"].nunique())
print("rebalance dates:", len(rebalance_dates), rebalance_dates.min().date(), "to", rebalance_dates.max().date())
print("final predictions:", predictions.shape)
print("prediction tickers:", predictions["ticker"].nunique())
print("SPY in predictions:", FEATURE_BENCHMARK_TICKER in set(predictions["ticker"]))
display(predictions.head())


secret price rows: 304562
SPY context rows: 3018
feature rows: 304562
all sequence predictions: (293457, 4)
test prediction dates before rebalance filter: 1002
rebalance dates: 201 2022-01-04 to 2025-12-30
final predictions: (26770, 4)
prediction tickers: 140
SPY in predictions: False


,date,ticker,horizon,expected_return
0,2022-01-04,ABBV,5,0.016900
1,2022-01-04,ABSI,5,0.019223
2,2022-01-04,ABUS,5,0.079874
3,2022-01-04,ACAD,5,0.060686
4,2022-01-04,ADMA,5,-0.040889


In [4]:
# Build Minho v2's submitted rank-long-only portfolio and backtest against Secret Set 2's XBI benchmark
portfolio = weights_from_predictions_rank_long_only(
    predictions,
    score_column="expected_return",
    dataset_name=SECRET_DATASET_NAME,
    strategy_name=f"{MINHO_V2_MODEL_NAME}_v2_{SECRET_DATASET_NAME}_{MINHO_V2_REBALANCE_FREQUENCY}",
)
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=SECRET_DATASET_NAME,
    repo_root=repo_root,
)

result = backtest_weights(
    SECRET_DATASET_NAME,
    portfolio,
    benchmark=backtest_benchmark_ticker,
    repo_root=repo_root,
)
metrics = build_metrics(result)

MINHO_V2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MINHO_V2_MODEL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
artifact_paths = write_backtest_artifacts(result, MINHO_V2_OUTPUT_DIR)

secret_predictions_path = MINHO_V2_OUTPUT_DIR / "predictions.parquet"
secret_features_path = MINHO_V2_OUTPUT_DIR / "features.parquet"
secret_metadata_path = MINHO_V2_OUTPUT_DIR / "secret_set_2_metadata.json"
predictions.to_parquet(secret_predictions_path, index=False)
secret_features.to_parquet(secret_features_path, index=False)

metadata_payload = {
    "model_name": MINHO_V2_MODEL_NAME,
    "source_mlflow_run_id": MINHO_V2_SOURCE_RUN_ID,
    "source_mlflow_experiment": MINHO_V2_SOURCE_EXPERIMENT,
    "source_dataset_name": MINHO_V2_SOURCE_DATASET_NAME,
    "secret_dataset_name": SECRET_DATASET_NAME,
    "feature_benchmark_ticker": FEATURE_BENCHMARK_TICKER,
    "backtest_benchmark_ticker": backtest_benchmark_ticker,
    "horizon": MINHO_V2_HORIZON,
    "seq_len": MINHO_V2_SEQ_LEN,
    "rebalance_frequency": MINHO_V2_REBALANCE_FREQUENCY,
    "feature_names": MINHO_V2_FEATURE_NAMES,
    "checkpoint_model_config": MINHO_V2_MODEL_CONFIG,
    "checkpoint_training": MINHO_V2_TRAINING_METADATA,
    "checkpoint_target": MINHO_V2_DATA_METADATA.get("target"),
}
secret_metadata_path.write_text(json.dumps(metadata_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

metrics_table = (
    pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(metrics.items())])
    .sort_values("metric")
    .reset_index(drop=True)
)

print("strategy:", portfolio.strategy_name)
print("weights:", validated_weights.shape)
print("backtest benchmark:", backtest_benchmark_ticker)
print("artifacts written to:", MINHO_V2_OUTPUT_DIR)
display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])


strategy: Minho_Baseline_LSTM_v2_secret_set_2_every_5_trading_days
weights: (201, 140)
backtest benchmark: XBI
artifacts written to: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/runs/minho_baseline_lstm_v2_secret_set_2_backtest


,metric,value
0,annual_excess_return_vs_benchmark,0.317503
1,annual_return,0.341612
2,annual_volatility,0.383330
3,average_turnover,0.231756
4,benchmark_annual_return,0.024109
5,benchmark_annual_volatility,0.328066
6,benchmark_max_drawdown,-0.435973
7,benchmark_sharpe,0.073488
8,benchmark_total_return,0.099693
9,calmar,0.687037


QuantStats report: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/runs/minho_baseline_lstm_v2_secret_set_2_backtest/quantstats.html


In [5]:
# Log Minho v2's Secret Set 2 backtest to MLflow
import mlflow
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient

mlflow_layout = init_mlflow(repo_root=repo_root)
mlflow.set_tracking_uri(mlflow_layout["tracking_uri"])

settings = load_mlflow_settings(repo_root)
experiment_name = f"{settings.experiment_prefix}_{SECRET_DATASET_NAME}"
client = MlflowClient()
if client.get_experiment_by_name(experiment_name) is None:
    for experiment in client.search_experiments(view_type=ViewType.DELETED_ONLY):
        if experiment.name == experiment_name:
            client.restore_experiment(experiment.experiment_id)
            print("Restored deleted MLflow experiment:", experiment_name)
            break

model_submission_config = {
    **MINHO_V2_MODEL_CONFIG,
    "seq_len": MINHO_V2_SEQ_LEN,
    "portfolio_builder": "weights_from_predictions_rank_long_only",
    "required_functions": ["build_model_features", "predict_from_prices"],
}

with start_run(
    run_name=f"{MINHO_V2_MODEL_NAME}_v2_secret_set_2_backtest",
    dataset_name=SECRET_DATASET_NAME,
    tags={
        "workflow": "minho_v2_secret_set_2_backtest",
        "model_family": "torch",
        "model_name": f"{MINHO_V2_MODEL_NAME}_v2",
        "source_mlflow_run_id": MINHO_V2_SOURCE_RUN_ID,
        "source_dataset_name": MINHO_V2_SOURCE_DATASET_NAME,
        "prediction_horizon": str(MINHO_V2_HORIZON),
        "rebalance_frequency": MINHO_V2_REBALANCE_FREQUENCY,
        "feature_benchmark": FEATURE_BENCHMARK_TICKER,
        "backtest_benchmark": backtest_benchmark_ticker,
    },
    repo_root=repo_root,
):
    mlflow.log_params(
        {
            "model_name": f"{MINHO_V2_MODEL_NAME}_v2",
            "dataset_name": SECRET_DATASET_NAME,
            "source_dataset_name": MINHO_V2_SOURCE_DATASET_NAME,
            "source_mlflow_run_id": MINHO_V2_SOURCE_RUN_ID,
            "horizon": MINHO_V2_HORIZON,
            "seq_len": MINHO_V2_SEQ_LEN,
            "feature_count": len(MINHO_V2_FEATURE_NAMES),
            "feature_list": ",".join(MINHO_V2_FEATURE_NAMES),
            "portfolio_builder": "weights_from_predictions_rank_long_only",
            "rebalance_frequency": MINHO_V2_REBALANCE_FREQUENCY,
            "feature_benchmark_ticker": FEATURE_BENCHMARK_TICKER,
            "backtest_benchmark_ticker": backtest_benchmark_ticker,
            "prediction_date_start": predictions["date"].min().date().isoformat(),
            "prediction_date_end": predictions["date"].max().date().isoformat(),
            "prediction_date_count": predictions["date"].nunique(),
            "prediction_ticker_count": predictions["ticker"].nunique(),
        }
    )
    mlflow.log_dict(metadata_payload, "secret_set_2_metadata.json")
    for path in [
        secret_predictions_path,
        secret_features_path,
        secret_metadata_path,
        MINHO_V2_SOURCE_MANIFEST_PATH,
        MINHO_V2_SOURCE_WEIGHTS_PATH,
        MINHO_V2_SOURCE_PREDICTIONS_PATH,
        MINHO_V2_SOURCE_DATASET_SPEC_PATH,
    ]:
        if Path(path).exists():
            mlflow.log_artifact(str(path), artifact_path="secret_set_2_artifacts")

    submission_manifest = log_model_submission(
        {"torch_model_v2": MINHO_V2_CHECKPOINT_PATH},
        model_name=f"{MINHO_V2_MODEL_NAME}_v2",
        model_family="torch",
        feature_names=MINHO_V2_FEATURE_NAMES,
        target=str(MINHO_V2_DATA_METADATA.get("target", f"forward_return_{MINHO_V2_HORIZON}d")),
        horizon=MINHO_V2_HORIZON,
        rebalance_frequency=MINHO_V2_REBALANCE_FREQUENCY,
        preprocessing=MINHO_V2_PREPROCESSING,
        model_config=model_submission_config,
        source_files=[MINHO_V2_NOTEBOOK_PATH, MINHO_V2_SOURCE_NOTEBOOK_ARTIFACT_PATH],
        notes="Minho newest sp500_subset_50 LSTM v2 checkpoint rerun on secret_set_2 with SPY feature context and XBI benchmark.",
    )
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print("Minho v2 Secret Set 2 MLflow logging complete.")
print(json.dumps(submission_manifest, indent=2))


🏃 View run Minho_Baseline_LSTM_v2_secret_set_2_backtest at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/6/runs/4838378481eb413c90d621249da8e8e2
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/6
Minho v2 Secret Set 2 MLflow logging complete.
{
  "model_name": "Minho_Baseline_LSTM_v2",
  "model_family": "torch",
  "target": "forward_alpha_5d_vs_spy",
  "horizon": 5,
  "rebalance_frequency": "every_5_trading_days",
  "feature_names": [
    "return_1d",
    "return_5d",
    "return_20d",
    "vol_5d",
    "vol_20d",
    "momentum_5d",
    "momentum_20d",
    "momentum_60d",
    "rsi_14",
    "macd_hist",
    "bollinger_z_20d",
    "beta_20d_spy",
    "excess_return_5d_vs_spy",
    "volume_zscore_20d",
    "price_to_sma_20d",
    "atr_14",
    "price_accel"
  ],
  "preprocessing": {
    "scaler": "train_mean_std",
    "means": {
      "return_1d": 0.0008256575024517243,
      "return_5d": 0.004020657054957735,
      "return_20d": 0.0158175752999

In [ ]:
# Final assertions for Minho v2's standalone Secret Set 2 test suite
assert not predictions.empty
assert not validated_weights.empty
assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert FEATURE_BENCHMARK_TICKER == "SPY"
assert backtest_benchmark_ticker == "XBI"
assert FEATURE_BENCHMARK_TICKER not in set(predictions["ticker"])
assert FEATURE_BENCHMARK_TICKER not in set(validated_weights.columns)
assert backtest_benchmark_ticker not in set(validated_weights.columns)
assert MINHO_V2_REBALANCE_FREQUENCY == "every_5_trading_days"
print("Minho v2 Secret Set 2 standalone test suite complete.")
